# Creating a Simple Agent with Tracing

In [1]:
import dotenv
import os

from openai import OpenAI

dotenv.load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    print(
        """Error: OPENAI_API_KEY environment variable not set. Please copy the .env.template file as .env and fill it in.
    
    You can execute these commands in the terminal to get started:
    cp .env.template .env
    code .env
    """
    )

# Test OpenAI Access
print(
    OpenAI()
    .responses.create(
        model=os.environ["OPENAI_DEFAULT_MODEL"], input="Say: We are up and running!"
    )
    .output_text
)

We are up and running!


In [2]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent

Create a simple Nutrition Assistant Agent

In [3]:
nutrition_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful assistant giving out nutrition advice.
    You give concise answers.
    """,
)

Let's execute the Agent:

In [4]:
with trace("Simple Nutrition Agent"):
    result = await Runner.run(nutrition_agent, "How healthy are bananas?")

print(result)

RunResult:
- Last agent: Agent(name="Nutrition Assistant", ...)
- Final output (str):
    Bananas are healthy in moderation. They’re a good source of:
    
    - Potassium (supports heart and muscle function)
    - Vitamin B6, vitamin C, and fiber
    - Carbs for quick energy; ripe bananas have more sugar and less resistant starch
    
    Tips:
    - One medium banana (~105 kcal) is a typical serving.
    - If watching sugar or blood sugar, pair with protein or fat (e.g., peanut butter, yogurt).
    - Unripe (green) bananas have more resistant starch; ripe ones have more sugar.
    
    Allergies are rare; otherwise, they fit well in most balanced diets.
- 2 new item(s)
- 1 raw response(s)
- 0 input guardrail result(s)
- 0 output guardrail result(s)
(See `RunResult` for more details)


Streaming the answer to the screen, token by token

In [5]:
response_stream = Runner.run_streamed(nutrition_agent, "How healthy are bananas?")

async for event in response_stream.stream_events():
    if event.type == "raw_response_event" and isinstance(
        event.data, ResponseTextDeltaEvent
    ):
        print(event.data.delta, end="", flush=True)

Bananas are overall healthy and convenient.

- Good stuff: potassium (supports heart function and blood pressure), vitamin C, vitamin B6, dietary fiber; low in fat and sodium.
- Calorie info: about 90 kcal for a medium banana.
- Considerations: natural sugars; portion matters for strict low-carb or diabetes plans—pair with protein or fat to slow absorption.
- Other notes: inexpensive, portable, widely available; allergy is rare.

Short answer: yes, a nutritious, balanced everyday fruit.

_Good Job!_

In [6]:
placement_trainer_agent = Agent(
    name="Placement Preparation Coach",
    instructions="""
    You are a helpful, practical guide for students preparing for job placements.
    Help with interview practice, resume improvements, aptitude and technical
    preparation, communication skills, and realistic study plans.

    Keep every answer concise, supportive, and specific. Give a short answer or
    a few actionable steps, and tailor advice to the student's target role,
    experience, and timeline when known. If important context is missing, ask
    at most one brief clarifying question. In mock interviews, ask one question
    at a time and give focused feedback after each answer. Never invent details
    about a company, job opening, or hiring process.
    """,
)

In [7]:
with trace("Simple Placement Preparation Coach"):
    result = await Runner.run(placement_trainer_agent, "How to prepare for an interview in 30 days?")

print(result)

RunResult:
- Last agent: Agent(name="Placement Preparation Coach", ...)
- Final output (str):
    Here’s a concise 30-day interview prep plan you can adapt to most roles.
    
    Week 1: Foundation
    - Clarify target role, two to three core skills, and your elevator pitch (150 words).
    - Polish resume and LinkedIn highlights to match target role.
    - Compile 5 STAR stories: challenges, actions, results (quantified).
    - Start daily coding/aptitude practice (30–45 min) if technical.
    - List 10 thoughtful questions to ask interviewers.
    
    Week 2: Technical and Portfolio
    - Deep-dive into 2–3 core topics (data structures for programmers; system design basics for seniors; frameworks/tools for other roles). Build or review a small project or case study.
    - Practice 5-7 common role-specific questions (coding problems, design questions, or case questions).
    - Do one mock interview focused on technical/problem-solving.
    - Refine resume bullets to show impact (use

In [8]:
StudentManager_agent = Agent(
    name="Student Manager",
    instructions="""
    You are a helpful assistant giving out academic advice.
    You give concise answers.
    """,
)

In [9]:
with trace("Simple Student Manager"):
    result = await Runner.run(StudentManager_agent, "How to prepare for an interview in 30 days?")

print(result)

RunResult:
- Last agent: Agent(name="Student Manager", ...)
- Final output (str):
    Here’s a concise 30-day interview prep plan you can follow.
    
     Week 1: Research & foundations
    - Day 1-2: target role/company research. Know products, metrics, competitors, culture.
    - Day 3: tailor your resume bullets to the role; map achievements to skills.
    - Day 4: compile 6-8 STAR stories (situations and results) from your experience.
    - Day 5: practice 5 behavioral questions (e.g., teamwork, conflict, leadership).
    - Day 6-7: learn the job’s core technical or domain skills you’ll be tested on.
    
     Week 2: Practice & structure
    - Day 8-9: practice 5-7 technical or case questions relevant to the role; use the STAR method for storytelling.
    - Day 10: mock interview with a friend/peer; record if possible.
    - Day 11-12: refine answers, shorten to 1–2 minutes per story; add measurable outcomes.
    - Day 13-14: prepare questions to ask the interviewer (culture, gro